# Notebook 04 — Cluster Analysis and Cluster Classifiers

---

## Objective

1. **Validate the clusters against the calendar.** Check whether the day-type and week-type clusters obtained in Notebook 03 are redundant with simple calendar variables (day of week, month, holiday) or whether they encode additional information.
2. **Train classifiers that predict the cluster of an observation from its features.** At forecast time the true cluster of a future hour is unknown, so the gating mechanism of the Mixture-of-Experts needs a classifier that predicts `cluster_day` and `cluster_week` from features that are available in advance. Six classifiers and five incremental feature sets are compared for each of the two targets, and the best combination per target is retrained on all data and saved.
3. **Analyse classifier confidence and calibration**, as input for the gating design of Notebook 06.

## Inputs

| File | Produced by | Content |
|---|---|---|
| `data/processed/features.parquet` | Notebook 02 | Hourly features (calendar, holiday, Fourier, demand lags, rolling-window statistics) |
| `data/clustering/cluster_assignments.parquet` | Notebook 03 | `datetime`, `cluster_day`, `cluster_week`, `expert_id` for every hour |

## Outputs

| File | Content | Used by |
|---|---|---|
| `models/classifiers/best_classifier_day.pkl` | Best `cluster_day` classifier (scikit-learn `Pipeline` with `StandardScaler`), its `LabelEncoder`, feature columns and configuration | Notebooks 06 and 08 |
| `models/classifiers/best_classifier_week.pkl` | Same for `cluster_week` | Notebooks 06 and 08 |
| `results/04_classifier_comparison.csv` | Cross-validated accuracy and macro-F1 for every (target, feature set, classifier) | — |
| `results/04_confidence_analysis.json` | Confidence statistics, suggested confidence thresholds and best configuration per target | — |
| `results/04_*.png` | Figures (cluster–calendar heatmaps, conditional distributions, classifier comparison, confidence, calibration and threshold analyses) | — |

## Place in the pipeline

01 data acquisition → 02 preprocessing / feature engineering → 03 clustering of daily and weekly profiles → **04 cluster analysis and cluster classifiers** → 05 expert model selection per (day cluster, week cluster) → 06 gating design (hard / soft / adaptive gating) → 07 hyperparameter optimization (HPO) → 08 production assembly → 09 benchmark.

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

---
## 1. Imports and configuration

Library imports, random seed, number of parallel jobs and the plotting style shared by all figures of this notebook.

In [ ]:
import os
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.metrics import (
    cohen_kappa_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED = 42
N_JOBS = -1

# ---- Plot style ----
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'serif',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_palette('colorblind')

np.random.seed(RANDOM_SEED)
print("Configuration loaded.")

---
## 2. Data loading

The feature table (Notebook 02) and the hourly cluster assignments (Notebook 03) are joined on their timestamp with an **inner join**, so only hours that have both features and a cluster assignment are kept (Notebook 03 restricted the clustering to 2022–2025 and discarded incomplete days and ISO weeks). Day of week, month and hour are extracted from the timestamp for the calendar analysis. Rows from 2020 and 2021 are then explicitly excluded, and the columns holding the day-type and week-type cluster labels are identified.

In [ ]:
# Load features
features_path = os.path.join(DATA_DIR, "processed", "features.parquet")
df_features = pd.read_parquet(features_path)
# Turn the 'datetime' index into a 'timestamp' column
df_features = df_features.reset_index().rename(columns={'datetime': 'timestamp'})
print(f"Features shape: {df_features.shape}")
print(f"Feature columns: {list(df_features.columns)}")

# Load cluster assignments
clusters_path = os.path.join(DATA_DIR, "clustering", "cluster_assignments.parquet")
df_clusters = pd.read_parquet(clusters_path)
# Each assignment carries its own timestamp (datetime column, see Notebook 03). Reconstructing it
# by position shifted it by 48 h, because the clustering discards the first, incomplete ISO week.
df_clusters = df_clusters.rename(columns={'datetime': 'timestamp'})

print(f"\nClusters shape: {df_clusters.shape}")
print(f"Cluster columns: {list(df_clusters.columns)}")

# Ensure timestamp is datetime in both tables
df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])
df_clusters['timestamp'] = pd.to_datetime(df_clusters['timestamp'])

# Join on timestamp
df = df_features.merge(df_clusters, on='timestamp', how='inner')
print(f"\nMerged shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Extract calendar variables for the analysis
df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0=Mon, 6=Sun
df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour

In [ ]:
# Identify cluster columns
cluster_cols = [c for c in df.columns if 'cluster' in c.lower()]
print(f"\nCluster columns found: {cluster_cols}")
df.head()

In [ ]:
# Exclude 2020 and 2021
df = df[~df["timestamp"].dt.year.isin([2020, 2021])]

In [ ]:
# Identify the cluster-assignment columns for day type and week type.
# Adapt these names to the actual content of the parquet file.
# Common patterns: cluster_day, cluster_week, cluster_type_day, cluster_type_week

day_cluster_col = None
week_cluster_col = None

for c in cluster_cols:
    cl = c.lower()
    if 'day' in cl and 'week' not in cl:
        day_cluster_col = c
    elif 'week' in cl:
        week_cluster_col = c

# Fallback: if no day/week column is recognized, use the first cluster column(s) for both analyses
if day_cluster_col is None and week_cluster_col is None and len(cluster_cols) >= 1:
    day_cluster_col = cluster_cols[0]
    week_cluster_col = cluster_cols[1] if len(cluster_cols) > 1 else cluster_cols[0]

CLUSTER_TARGETS = {}
if day_cluster_col:
    CLUSTER_TARGETS['type_day'] = day_cluster_col
if week_cluster_col:
    CLUSTER_TARGETS['type_week'] = week_cluster_col

print(f"Cluster targets: {CLUSTER_TARGETS}")
for name, col in CLUSTER_TARGETS.items():
    print(f"  {name} ({col}): {df[col].nunique()} clusters, distribution:")
    print(f"    {df[col].value_counts().sort_index().to_dict()}")

In [ ]:
df

---

## 3. Cluster vs. calendar variables

This section evaluates whether the cluster assignments are redundant with the calendar variables **day of week**, **month** and **holiday** (`is_holiday`). If the clusters were essentially a relabelling of the calendar, a calendar-based routing would suffice and the clustering would add little value to the Mixture-of-Experts. For each clustering (`type_day`, `type_week`) and each calendar variable, three association measures are computed on the hourly observations:

- **Cohen's kappa** between the cluster labels and the calendar labels;
- **Normalized Mutual Information (NMI)**, which is invariant to permutations of the labels;
- **Pearson's χ² test of independence** on the contingency table (with a large number of observations even weak dependencies are significant, so effect sizes are emphasized over p-values).

Interpretation used in this notebook: if kappa < 0.3 and NMI is close to zero, the clusters encode information **beyond** the calendar variables (e.g., weather conditions, economic activity or behavioural patterns); if kappa > 0.5, the clusters are partially redundant with the calendar variables.

*Note.* Cohen's kappa compares the two labelings value by value (cluster `0` with calendar value `0`, etc.), so it is only meaningful when both label sets share a common coding; cluster IDs are arbitrary. NMI and the χ² test do not depend on the label coding. The row-normalized contingency heatmaps and conditional distributions that follow give a direct visual view of the association.

In [ ]:
# ---- Cohen's kappa, NMI and chi-square for each clustering vs. calendar variables ----

calendar_vars = {
    'day_of_week': df['day_of_week'],
    'month': df['month'],
    'is_holiday': df['is_holiday'],
}

agreement_results = []

for clustering_name, cluster_col in CLUSTER_TARGETS.items():
    cluster_values = df[cluster_col]

    for cal_name, cal_values in calendar_vars.items():
        # Cohen's kappa -- computed even when the two variables have different cardinalities
        # (labels are compared value by value)
        kappa = cohen_kappa_score(cluster_values, cal_values)

        # Normalized mutual information
        nmi = normalized_mutual_info_score(cluster_values, cal_values)

        # Chi-square test of independence
        contingency = pd.crosstab(cluster_values, cal_values)
        chi2, p_value, dof, expected = chi2_contingency(contingency)

        agreement_results.append({
            'clustering': clustering_name,
            'calendar_var': cal_name,
            'kappa': kappa,
            'nmi': nmi,
            'chi2': chi2,
            'chi2_p_value': p_value,
            'dof': dof,
        })

df_agreement = pd.DataFrame(agreement_results)
print("=" * 80)
print("AGREEMENT SUMMARY: CLUSTER VS. CALENDAR VARIABLE")
print("=" * 80)
display(df_agreement.style.format({
    'kappa': '{:.4f}',
    'nmi': '{:.4f}',
    'chi2': '{:.1f}',
    'chi2_p_value': '{:.2e}',
}).set_caption('Agreement Metrics between Clusters and Calendar Variables'))

In [ ]:
# ---- Interpretation guide ----

print("\nINTERPRETATION GUIDE:")
print("-" * 60)
print("Cohen's kappa:")
print("  < 0.0  : Less than chance agreement")
print("  0.0-0.2: Slight agreement")
print("  0.2-0.4: Fair agreement")
print("  0.4-0.6: Moderate agreement")
print("  0.6-0.8: Substantial agreement")
print("  0.8-1.0: Almost perfect agreement")
print("\nNMI (Normalized Mutual Information):")
print("  0.0: Independent (no shared information)")
print("  1.0: Perfect correspondence")
print("  < 0.1: Negligible dependence")
print("\nChi-square test:")
print("  p < 0.05: Statistical dependence (independence is rejected)")
print("  Note: with large N, even small effects are significant.")
print("  Focus on the effect size (kappa, NMI), not only on the p-value.")

print("\n" + "=" * 60)
print("KEY FINDINGS:")
print("=" * 60)
for _, row in df_agreement.iterrows():
    kappa_interp = (
        "not redundant" if abs(row['kappa']) < 0.3
        else "partially redundant" if abs(row['kappa']) < 0.5
        else "highly redundant"
    )
    print(f"  {row['clustering']} vs {row['calendar_var']}: "
          f"kappa={row['kappa']:.3f} ({kappa_interp}), NMI={row['nmi']:.3f}")

In [ ]:
# ---- Cluster vs. day of week (row-normalized contingency heatmaps) ----

dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(7 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, (clustering_name, cluster_col) in zip(axes, CLUSTER_TARGETS.items()):
    ct = pd.crosstab(
        df[cluster_col],
        df['day_of_week'],
        normalize='index',  # Row-normalize: distribution across days FOR each cluster
    )
    ct.columns = [dow_labels[i] for i in ct.columns]

    sns.heatmap(
        ct, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
        cbar_kws={'label': 'Proportion'},
        linewidths=0.5,
    )
    ax.set_title(f'{clustering_name}: Cluster vs Day of Week\n(row-normalized)', fontweight='bold')
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Cluster ID')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_cluster_vs_dow_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("If rows look uniform, clusters do NOT correspond to specific days -- good for non-redundancy.")

In [ ]:
# ---- Cluster vs. month (row-normalized contingency heatmaps) ----

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(8 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, (clustering_name, cluster_col) in zip(axes, CLUSTER_TARGETS.items()):
    ct = pd.crosstab(
        df[cluster_col],
        df['month'],
        normalize='index',
    )
    ct.columns = [month_labels[i - 1] for i in ct.columns]

    sns.heatmap(
        ct, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax,
        cbar_kws={'label': 'Proportion'},
        linewidths=0.5,
    )
    ax.set_title(f'{clustering_name}: Cluster vs Month\n(row-normalized)', fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Cluster ID')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_cluster_vs_month_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Distribution of clusters conditioned on each calendar variable ----

for clustering_name, cluster_col in CLUSTER_TARGETS.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{clustering_name}: Cluster Distribution Conditioned on Calendar Variables',
                 fontweight='bold', fontsize=14, y=1.02)

    # (a) Conditioned on Day of Week
    ct_dow = pd.crosstab(df['day_of_week'], df[cluster_col], normalize='index')
    ct_dow.index = [dow_labels[i] for i in ct_dow.index]
    ct_dow.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[0].set_title('By Day of Week')
    axes[0].set_xlabel('Day of Week')
    axes[0].set_ylabel('Cluster Proportion')
    axes[0].legend(title='Cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

    # (b) Conditioned on Month
    ct_month = pd.crosstab(df['month'], df[cluster_col], normalize='index')
    ct_month.index = [month_labels[i - 1] for i in ct_month.index]
    ct_month.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[1].set_title('By Month')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Cluster Proportion')
    axes[1].legend(title='Cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

    # (c) Conditioned on Holiday
    ct_hol = pd.crosstab(df['is_holiday'], df[cluster_col], normalize='index')
    ct_hol.index = ['Non-Holiday', 'Holiday']
    ct_hol.plot(kind='bar', stacked=True, ax=axes[2], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[2].set_title('By Holiday Status')
    axes[2].set_xlabel('Holiday')
    axes[2].set_ylabel('Cluster Proportion')
    axes[2].legend(title='Cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_cluster_dist_conditioned_{clustering_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

---

## 4. Cluster classifiers with multiple feature-set configurations

The gating mechanism must infer the cluster of an hour from its features. Classifiers are therefore trained to predict cluster membership (`type_day` → `cluster_day`, `type_week` → `cluster_week`) using progressively richer feature sets, in order to determine which combination of features and algorithm is best suited for the gating.

### 4.1 Feature sets

| Feature set | Content |
|---|---|
| `calendar_only` | Cyclical hour, day-of-week and month encodings (sin/cos), `is_holiday`, `is_bridge` (bridge holiday), days to the next / from the previous holiday |
| `calendar_fourier` | `calendar_only` + Fourier terms (`fourier_*`) |
| `calendar_lags` | `calendar_only` + demand lags (`demand_lag_*`) |
| `calendar_lags_rolling` | `calendar_lags` + rolling-window statistics (`*rolling*`) |
| `all_production` | Calendar + Fourier + lags + rolling-window features |

### 4.2 Classifiers

Logistic regression (multinomial), random forest, gradient boosting (scikit-learn), XGBoost, k-nearest neighbours ($k = 7$) and an RBF support vector classifier, with the fixed hyperparameters defined below (no hyperparameter tuning is performed in this notebook). Every classifier is wrapped in a `Pipeline` with a `StandardScaler`, which is fit on the training folds only.

### 4.3 Validation

Rows with a missing value in any feature of the largest set (`all_production`, typically the first hours of the series because of lags and rolling windows) are dropped, so that all feature sets are evaluated on exactly the same observations. Each (target, feature set, classifier) combination is evaluated by **10-fold stratified cross-validation with shuffling** (`StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`), reporting the mean and standard deviation of accuracy and macro-averaged F1 over the folds, and the mean maximum predicted probability (mean confidence) on the validation folds (left as NaN for classifiers that do not expose `predict_proba`; with `probability=False`, the default, this is the case for `SVC`).

*Note.* Because the folds are shuffled at the hourly level, hours of the same day or week (which share the same cluster label and highly correlated lag/rolling features) can appear in both the training and validation folds. The cross-validated scores should therefore be read as a relative comparison between configurations rather than as an estimate of out-of-sample performance on future periods. (The variable holding the splitter is named `tscv`, but it is a `StratifiedKFold`, not a `TimeSeriesSplit`.)

In [ ]:
# ---- Define feature sets ----

# Identify the available feature columns by pattern matching
all_cols = list(df.columns)

calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

# Fourier features
fourier_features = sorted([c for c in all_cols if c.startswith('fourier_')])

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

# Rolling-window features
rolling_features = sorted([c for c in all_cols if 'rolling' in c])

# Check that all features exist
for name, flist in [('calendar', calendar_only), ('fourier', fourier_features),
                     ('lag', lag_features), ('rolling', rolling_features)]:
    missing = [f for f in flist if f not in all_cols]
    if missing:
        print(f"WARNING: missing {name} features: {missing}")
    else:
        print(f"{name}: {len(flist)} features OK")

# Build incremental feature sets
feature_sets = {
    'calendar_only': calendar_only,
    'calendar_fourier': calendar_only + fourier_features,
    'calendar_lags': calendar_only + lag_features,
    'calendar_lags_rolling': calendar_only + lag_features + rolling_features,
    'all_production': calendar_only + fourier_features + lag_features + rolling_features,
}

for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")

In [ ]:
# ---- Define classifiers ----
classifiers = {
    'LogisticRegression': LogisticRegression(
        max_iter=2000, multi_class='multinomial', random_state=RANDOM_SEED,
        C=1.0, n_jobs=N_JOBS
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        random_state=RANDOM_SEED, n_jobs=N_JOBS
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5,
        learning_rate=0.1, random_state=RANDOM_SEED
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric='mlogloss', verbosity=0, random_state=RANDOM_SEED),
    'KNeighbors': KNeighborsClassifier(
        n_neighbors=7, n_jobs=N_JOBS
    ),
    'SVC': SVC(
        kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_SEED
    ),
}

print(f"Classifiers: {list(classifiers.keys())}")
print(f"Cluster targets: {list(CLUSTER_TARGETS.keys())}")
print(f"Feature sets: {list(feature_sets.keys())}")
n_combos = len(classifiers) * len(feature_sets) * len(CLUSTER_TARGETS)
print(f"Total combinations: {n_combos}")

In [ ]:
# ---- Run the cross-validated classification experiments ----
tscv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)  # shuffled stratified K-fold (not a time-series split)

# Drop rows with NaN in any feature (caused by lags/rolling windows at the start of the series)
all_feature_cols = feature_sets['all_production']
valid_mask = df[all_feature_cols].notna().all(axis=1)
df_valid = df[valid_mask].reset_index(drop=True)
print(f"Valid rows after dropping NaN: {len(df_valid)} / {len(df)} ({len(df_valid)/len(df)*100:.1f}%)")

results = []

for target_name, cluster_col in CLUSTER_TARGETS.items():
    y = df_valid[cluster_col].values
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    n_classes = len(le.classes_)
    print(f"\n{'='*70}")
    print(f"TARGET: {target_name} ({cluster_col}) -- {n_classes} classes")
    print(f"{'='*70}")

    for fs_name, fs_cols in feature_sets.items():
        # Keep only the columns that exist in df_valid
        fs_cols_valid = [c for c in fs_cols if c in df_valid.columns]
        X = df_valid[fs_cols_valid].values

        for clf_name, clf_template in classifiers.items():
            fold_accs = []
            fold_f1s = []
            fold_probas = []  # Store max(predict_proba) of every validation sample

            for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X, y_encoded)):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

                # Pipeline with scaling (important for LR, KNN, SVC)
                from sklearn.base import clone
                clf = clone(clf_template)
                pipe = Pipeline([
                    ('scaler', StandardScaler()),
                    ('clf', clf),
                ])

                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)

                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
                fold_accs.append(acc)
                fold_f1s.append(f1)

                # Confidence: maximum of predict_proba
                if hasattr(pipe.named_steps['clf'], 'predict_proba'):
                    probas = pipe.predict_proba(X_test)
                    max_proba = probas.max(axis=1)
                    fold_probas.extend(max_proba.tolist())

            mean_acc = np.mean(fold_accs)
            std_acc = np.std(fold_accs)
            mean_f1 = np.mean(fold_f1s)
            std_f1 = np.std(fold_f1s)
            mean_confidence = np.mean(fold_probas) if fold_probas else np.nan

            results.append({
                'target': target_name,
                'feature_set': fs_name,
                'classifier': clf_name,
                'accuracy_mean': mean_acc,
                'accuracy_std': std_acc,
                'f1_macro_mean': mean_f1,
                'f1_macro_std': std_f1,
                'mean_confidence': mean_confidence,
                'n_features': len(fs_cols_valid),
            })

            print(f"  {fs_name:25s} | {clf_name:20s} | "
                  f"Acc={mean_acc:.4f}+/-{std_acc:.4f} | "
                  f"F1={mean_f1:.4f}+/-{std_f1:.4f}")

df_results = pd.DataFrame(results)
print(f"\nTotal experiments: {len(df_results)}")

In [ ]:
# ---- Summary table: feature_set x classifier -> metrics per target ----

for target_name in CLUSTER_TARGETS:
    print(f"\n{'='*80}")
    print(f"SUMMARY TABLE: {target_name}")
    print(f"{'='*80}")

    subset = df_results[df_results['target'] == target_name]

    # Pivot tables for accuracy and macro F1
    pivot_acc = subset.pivot_table(
        index='feature_set', columns='classifier',
        values='accuracy_mean', aggfunc='first'
    )
    pivot_f1 = subset.pivot_table(
        index='feature_set', columns='classifier',
        values='f1_macro_mean', aggfunc='first'
    )

    # Order the feature sets logically
    fs_order = ['calendar_only', 'calendar_fourier', 'calendar_lags',
                'calendar_lags_rolling', 'all_production']
    fs_order = [f for f in fs_order if f in pivot_acc.index]

    print("\n--- Accuracy ---")
    display(pivot_acc.loc[fs_order].style.format('{:.4f}').background_gradient(
        cmap='YlGn', axis=None
    ).set_caption(f'Accuracy: {target_name}'))

    print("\n--- F1 Macro ---")
    display(pivot_f1.loc[fs_order].style.format('{:.4f}').background_gradient(
        cmap='YlGn', axis=None
    ).set_caption(f'F1 Macro: {target_name}'))

In [ ]:
# ---- Visual comparison: accuracy and F1 per feature set and classifier ----

for target_name in CLUSTER_TARGETS:
    subset = df_results[df_results['target'] == target_name].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Classifier Comparison: {target_name}', fontweight='bold', fontsize=14)

    fs_order = ['calendar_only', 'calendar_fourier', 'calendar_lags',
                'calendar_lags_rolling', 'all_production']
    subset['feature_set'] = pd.Categorical(subset['feature_set'], categories=fs_order, ordered=True)
    subset = subset.sort_values('feature_set')

    # Accuracy
    sns.barplot(
        data=subset, x='feature_set', y='accuracy_mean', hue='classifier',
        ax=axes[0], edgecolor='black', linewidth=0.5,
    )
    axes[0].set_title('Accuracy (mean over CV folds)')
    axes[0].set_xlabel('Feature Set')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
    axes[0].legend(title='Classifier', fontsize=8, loc='lower right')

    # F1 Macro
    sns.barplot(
        data=subset, x='feature_set', y='f1_macro_mean', hue='classifier',
        ax=axes[1], edgecolor='black', linewidth=0.5,
    )
    axes[1].set_title('F1 Macro (mean over CV folds)')
    axes[1].set_xlabel('Feature Set')
    axes[1].set_ylabel('F1 Macro')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
    axes[1].legend(title='Classifier', fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_classifier_comparison_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

### 4.4 Selection and final training

For each target, configurations are ranked by mean macro-F1 (accuracy as tie-breaker), and the best (feature set, classifier) pair is selected. The selected pipeline is then refit on **all valid observations** (no hold-out). The classification report printed below is therefore **in-sample (training) performance** and is not an estimate of generalization error; the cross-validated scores above are the relevant comparison metrics.

In [ ]:
# ---- Identify the best classifier for each target ----

# Ranking criterion: highest macro F1, then accuracy as tie-breaker
best_configs = {}

for target_name in CLUSTER_TARGETS:
    subset = df_results[df_results['target'] == target_name].copy()
    subset = subset.sort_values(
        ['f1_macro_mean', 'accuracy_mean'], ascending=[False, False]
    )
    best_row = subset.iloc[0]
    best_configs[target_name] = {
        'classifier': best_row['classifier'],
        'feature_set': best_row['feature_set'],
        'accuracy': best_row['accuracy_mean'],
        'f1_macro': best_row['f1_macro_mean'],
        'mean_confidence': best_row['mean_confidence'],
    }

    print(f"\nBEST for {target_name}:")
    print(f"  Classifier: {best_row['classifier']}")
    print(f"  Feature set: {best_row['feature_set']}")
    print(f"  Accuracy: {best_row['accuracy_mean']:.4f} +/- {best_row['accuracy_std']:.4f}")
    print(f"  F1 Macro: {best_row['f1_macro_mean']:.4f} +/- {best_row['f1_macro_std']:.4f}")
    print(f"  Mean confidence: {best_row['mean_confidence']:.4f}")

    # Show top 5
    print(f"\n  Top 5 configurations:")
    display(subset.head(5)[[
        'feature_set', 'classifier', 'accuracy_mean', 'f1_macro_mean', 'mean_confidence'
    ]].reset_index(drop=True))

In [ ]:
# ---- Train the final (best) classifiers on all data ----
# (These are saved for production use within the gating mechanism)

final_classifiers = {}
final_label_encoders = {}

for target_name, cluster_col in CLUSTER_TARGETS.items():
    config = best_configs[target_name]
    fs_cols = feature_sets[config['feature_set']]
    fs_cols_valid = [c for c in fs_cols if c in df_valid.columns]

    X_full = df_valid[fs_cols_valid].values
    y_full = df_valid[cluster_col].values
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_full)

    clf_template = classifiers[config['classifier']]
    from sklearn.base import clone
    clf = clone(clf_template)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', clf),
    ])
    pipe.fit(X_full, y_encoded)

    final_classifiers[target_name] = {
        'pipeline': pipe,
        'label_encoder': le,
        'feature_columns': fs_cols_valid,
        'config': config,
    }
    final_label_encoders[target_name] = le

    # Classification report on all data (training performance)
    y_pred_full = pipe.predict(X_full)
    print(f"\n{'='*60}")
    print(f"FINAL MODEL - {target_name} ({config['classifier']} + {config['feature_set']})")
    print(f"{'='*60}")
    print(classification_report(
        y_encoded, y_pred_full,
        target_names=[str(c) for c in le.classes_],
    ))

print("Final classifiers trained on the full dataset.")

---

## 5. Classifier confidence analysis

Understanding when the classifier is uncertain is critical for the gating mechanism: low-confidence predictions correspond to observations that do not clearly belong to a single cluster — expected to occur, for instance, on holidays, bridge holidays or seasonal transitions. Here, the **confidence** of a prediction is the maximum predicted class probability, $\max_c \hat{p}(c \mid x)$.

For each target, using the final classifiers of Section 4.4, this section computes:

1. summary statistics of the confidence (mean, median, quantiles and fraction of predictions below 0.5, 0.6, 0.7 and 0.8) and its histogram;
2. confidence broken down by holiday status, day of week, bridge-holiday status and hour of day;
3. **calibration curves** (one-vs-rest, per class, 10 uniform bins) and a **reliability diagram** (accuracy vs. confidence in 10 bins, together with the number of samples per bin);
4. a **suggested confidence threshold**: for thresholds $t \in \{0.30, 0.35, \dots, 0.90\}$, the accuracy on observations with confidence $\ge t$ and the fraction of retained observations are computed; the suggested threshold is the one with the highest accuracy among those retaining at least 90% of the observations.

All quantities in this section are computed on the same observations used to fit the final classifiers (in-sample), so they describe the fitted models rather than out-of-sample behaviour. This analysis is an input for the design of the gating mechanism (Notebook 06).

In [ ]:
# ---- Confidence distribution of the final classifiers ----

confidence_analysis = {}

for target_name, model_info in final_classifiers.items():
    pipe = model_info['pipeline']
    fs_cols_valid = model_info['feature_columns']

    X_full = df_valid[fs_cols_valid].values
    probas = pipe.predict_proba(X_full)
    max_proba = probas.max(axis=1)

    df_valid[f'confidence_{target_name}'] = max_proba
    df_valid[f'predicted_{target_name}'] = pipe.predict(X_full)

    # Statistics
    conf_stats = {
        'mean': float(np.mean(max_proba)),
        'median': float(np.median(max_proba)),
        'std': float(np.std(max_proba)),
        'q05': float(np.percentile(max_proba, 5)),
        'q10': float(np.percentile(max_proba, 10)),
        'q25': float(np.percentile(max_proba, 25)),
        'q75': float(np.percentile(max_proba, 75)),
        'q90': float(np.percentile(max_proba, 90)),
        'pct_below_50': float((max_proba < 0.5).mean()),
        'pct_below_60': float((max_proba < 0.6).mean()),
        'pct_below_70': float((max_proba < 0.7).mean()),
        'pct_below_80': float((max_proba < 0.8).mean()),
    }
    confidence_analysis[target_name] = conf_stats

    print(f"\nConfidence statistics - {target_name}:")
    for k, v in conf_stats.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
# ---- Histogram of max(predict_proba) ----

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(7 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, target_name in zip(axes, CLUSTER_TARGETS):
    conf_col = f'confidence_{target_name}'
    conf_values = df_valid[conf_col]

    ax.hist(conf_values, bins=50, edgecolor='black', linewidth=0.5, alpha=0.8, color='steelblue')

    # Mark percentiles
    for pct, color, ls in [(10, 'red', '--'), (25, 'orange', '-.'), (50, 'green', ':')]:
        val = np.percentile(conf_values, pct)
        ax.axvline(val, color=color, linestyle=ls, linewidth=1.5, label=f'P{pct}={val:.2f}')

    ax.set_title(f'{target_name}: Distribution of max(predict_proba)', fontweight='bold')
    ax.set_xlabel('Max Predicted Probability (Confidence)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_confidence_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Confidence by observation type ----
# When is the classifier less confident? Expected: holidays, bridge holidays, transitions.

for target_name in CLUSTER_TARGETS:
    conf_col = f'confidence_{target_name}'

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{target_name}: Confidence Analysis by Observation Type',
                 fontweight='bold', fontsize=14)

    # (a) Confidence by holiday status
    groups = df_valid.groupby('is_holiday')[conf_col]
    data_holiday = [groups.get_group(0).values, groups.get_group(1).values]
    bp = axes[0, 0].boxplot(data_holiday, labels=['Non-Holiday', 'Holiday'],
                            patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('salmon')
    axes[0, 0].set_title('Confidence by Holiday Status')
    axes[0, 0].set_ylabel('Max Predicted Probability')

    # (b) Confidence by day of week
    dow_data = [df_valid[df_valid['day_of_week'] == d][conf_col].values for d in range(7)]
    bp2 = axes[0, 1].boxplot(dow_data, labels=dow_labels, patch_artist=True, widths=0.6)
    colors_dow = plt.cm.Set3(np.linspace(0, 1, 7))
    for box, color in zip(bp2['boxes'], colors_dow):
        box.set_facecolor(color)
    axes[0, 1].set_title('Confidence by Day of Week')
    axes[0, 1].set_ylabel('Max Predicted Probability')
    axes[0, 1].set_xticklabels(dow_labels, rotation=45, ha='right')

    # (c) Confidence by bridge status
    if 'is_bridge' in df_valid.columns:
        groups_bridge = df_valid.groupby('is_bridge')[conf_col]
        data_bridge = [groups_bridge.get_group(0).values, groups_bridge.get_group(1).values]
        bp3 = axes[1, 0].boxplot(data_bridge, labels=['Non-Bridge', 'Bridge'],
                                  patch_artist=True, widths=0.5)
        bp3['boxes'][0].set_facecolor('lightblue')
        bp3['boxes'][1].set_facecolor('lightyellow')
        axes[1, 0].set_title('Confidence by Bridge Day Status')
        axes[1, 0].set_ylabel('Max Predicted Probability')

    # (d) Confidence by hour of day
    hour_means = df_valid.groupby('hour')[conf_col].mean()
    hour_stds = df_valid.groupby('hour')[conf_col].std()
    axes[1, 1].fill_between(
        hour_means.index, hour_means - hour_stds, hour_means + hour_stds,
        alpha=0.3, color='steelblue',
    )
    axes[1, 1].plot(hour_means.index, hour_means, 'o-', color='steelblue', markersize=4)
    axes[1, 1].set_title('Mean Confidence by Hour of Day')
    axes[1, 1].set_xlabel('Hour')
    axes[1, 1].set_ylabel('Mean Max Predicted Probability')
    axes[1, 1].set_xticks(range(0, 24, 2))

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_confidence_by_obs_type_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Calibration curves and reliability diagrams ----

for target_name, model_info in final_classifiers.items():
    pipe = model_info['pipeline']
    le = model_info['label_encoder']
    fs_cols_valid = model_info['feature_columns']
    cluster_col = CLUSTER_TARGETS[target_name]

    X_full = df_valid[fs_cols_valid].values
    y_true = le.transform(df_valid[cluster_col].values)
    probas = pipe.predict_proba(X_full)
    n_classes = probas.shape[1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'{target_name}: Calibration Analysis', fontweight='bold', fontsize=14)

    # (a) Per-class calibration curves
    for cls_idx in range(n_classes):
        y_binary = (y_true == cls_idx).astype(int)
        prob_cls = probas[:, cls_idx]

        if y_binary.sum() < 10:
            continue

        fraction_pos, mean_predicted = calibration_curve(
            y_binary, prob_cls, n_bins=10, strategy='uniform'
        )
        axes[0].plot(mean_predicted, fraction_pos, 's-',
                     label=f'Cluster {le.classes_[cls_idx]}', markersize=4)

    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    axes[0].set_title('Calibration Curve (per class)')
    axes[0].set_xlabel('Mean Predicted Probability')
    axes[0].set_ylabel('Fraction of Positives')
    axes[0].legend(fontsize=8)
    axes[0].set_xlim([-0.02, 1.02])
    axes[0].set_ylim([-0.02, 1.02])

    # (b) Reliability diagram: confidence vs actual accuracy
    max_proba = probas.max(axis=1)
    y_pred = probas.argmax(axis=1)
    correct = (y_pred == y_true).astype(int)

    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_accs = []
    bin_counts = []

    for i in range(n_bins):
        mask = (max_proba >= bin_edges[i]) & (max_proba < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_accs.append(correct[mask].mean())
            bin_counts.append(mask.sum())

    ax_bar = axes[1]
    ax_bar2 = ax_bar.twinx()

    bars = ax_bar.bar(bin_centers, bin_accs, width=0.08, alpha=0.7,
                      color='steelblue', edgecolor='black', linewidth=0.5, label='Accuracy')
    ax_bar.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax_bar.set_title('Reliability Diagram')
    ax_bar.set_xlabel('Mean Confidence (max probability)')
    ax_bar.set_ylabel('Actual Accuracy', color='steelblue')
    ax_bar.set_xlim([-0.02, 1.02])
    ax_bar.set_ylim([-0.02, 1.02])
    ax_bar.legend(loc='upper left', fontsize=8)

    ax_bar2.bar(bin_centers, bin_counts, width=0.04, alpha=0.4,
                color='orange', edgecolor='black', linewidth=0.3, label='Count')
    ax_bar2.set_ylabel('Sample Count', color='orange')
    ax_bar2.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_calibration_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Suggest a confidence threshold ----
# For each candidate threshold t, compute the accuracy on the observations with confidence >= t
# and the fraction of observations retained. The suggested threshold is the one with the highest
# accuracy among those that retain at least 90% of the data.

for target_name in CLUSTER_TARGETS:
    conf_col = f'confidence_{target_name}'
    pred_col = f'predicted_{target_name}'
    cluster_col = CLUSTER_TARGETS[target_name]

    le = final_classifiers[target_name]['label_encoder']
    y_true = le.transform(df_valid[cluster_col].values)
    y_pred = df_valid[pred_col].values
    conf_values = df_valid[conf_col].values
    correct = (y_pred == y_true)

    thresholds = np.arange(0.3, 0.95, 0.05)
    results_thresh = []
    for t in thresholds:
        mask_above = conf_values >= t
        if mask_above.sum() > 0:
            acc_above = correct[mask_above].mean()
            pct_above = mask_above.mean()
        else:
            acc_above = np.nan
            pct_above = 0.0
        results_thresh.append({
            'threshold': t,
            'accuracy_above': acc_above,
            'pct_retained': pct_above,
        })

    df_thresh = pd.DataFrame(results_thresh)

    # Suggested threshold: retain at least 90% of data with best accuracy
    viable = df_thresh[df_thresh['pct_retained'] >= 0.90]
    if len(viable) > 0:
        best_thresh = viable.loc[viable['accuracy_above'].idxmax(), 'threshold']
    else:
        best_thresh = df_thresh.loc[df_thresh['pct_retained'].idxmax(), 'threshold']

    confidence_analysis[target_name]['suggested_threshold'] = float(best_thresh)

    print(f"\n{target_name}: Threshold Analysis")
    print(f"  Suggested threshold: {best_thresh:.2f}")
    display(df_thresh.style.format({
        'threshold': '{:.2f}',
        'accuracy_above': '{:.4f}',
        'pct_retained': '{:.4f}',
    }).set_caption(f'{target_name}: Accuracy vs Confidence Threshold'))

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    l1 = ax1.plot(df_thresh['threshold'], df_thresh['accuracy_above'], 'o-',
                  color='steelblue', label='Accuracy (above threshold)', markersize=5)
    l2 = ax2.plot(df_thresh['threshold'], df_thresh['pct_retained'], 's-',
                  color='coral', label='% Retained', markersize=5)

    ax1.axvline(best_thresh, color='green', linestyle='--', linewidth=1.5,
                label=f'Suggested: {best_thresh:.2f}')

    ax1.set_xlabel('Confidence Threshold')
    ax1.set_ylabel('Accuracy', color='steelblue')
    ax2.set_ylabel('% Data Retained', color='coral')
    ax1.set_title(f'{target_name}: Accuracy vs. Confidence Threshold', fontweight='bold')

    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='lower left', fontsize=9)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_threshold_analysis_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

---

## 6. Saving artifacts

The selected classifiers are persisted for use in the gating mechanism (Notebook 06) and in the production assembly (Notebook 08), together with the full comparison table and the confidence-analysis metadata:

| File | Content |
|---|---|
| `models/classifiers/best_classifier_day.pkl` | Dictionary with `pipeline` (StandardScaler + classifier), `label_encoder`, `feature_columns`, `config`, `cluster_column`, `target_name` for `type_day` |
| `models/classifiers/best_classifier_week.pkl` | Same for `type_week` |
| `results/04_classifier_comparison.csv` | Cross-validation results for every (target, feature set, classifier) |
| `results/04_confidence_analysis.json` | Confidence statistics and suggested thresholds (`analysis`) and the best configuration per target (`best_classifiers`) |

In [ ]:
# ---- Save classifiers ----

classifiers_dir = os.path.join(MODELS_DIR, 'classifiers')  # created by the setup cell

for target_name, model_info in final_classifiers.items():
    # Determine the file name
    if 'day' in target_name:
        fname = 'best_classifier_day.pkl'
    else:
        fname = 'best_classifier_week.pkl'

    save_path = os.path.join(classifiers_dir, fname)

    artifact = {
        'pipeline': model_info['pipeline'],
        'label_encoder': model_info['label_encoder'],
        'feature_columns': model_info['feature_columns'],
        'config': model_info['config'],
        'cluster_column': CLUSTER_TARGETS[target_name],
        'target_name': target_name,
    }

    with open(save_path, 'wb') as f:
        pickle.dump(artifact, f)

    print(f"Saved: {save_path}")

# ---- Save the comparison table ----
results_csv_path = os.path.join(RESULTS_DIR, '04_classifier_comparison.csv')
df_results.to_csv(results_csv_path, index=False)
print(f"Saved: {results_csv_path}")

# ---- Save the confidence analysis ----
conf_json_path = os.path.join(RESULTS_DIR, '04_confidence_analysis.json')

# Add the best-classifier information
confidence_output = {
    'analysis': confidence_analysis,
    'best_classifiers': best_configs,
}

with open(conf_json_path, 'w') as f:
    json.dump(confidence_output, f, indent=2)
print(f"Saved: {conf_json_path}")

print("\nAll files saved successfully.")